# Bronze Layer Ingestion

This notebook ingests the original banking CSV files using PySpark.

## Objective

Load source data into the Bronze layer while preserving the original structure and values.

## Source files

- accounts.csv
- branches.csv
- cards.csv
- customers.csv
- loans.csv
- merchants.csv


In [27]:
print("Notebook funcionando correctamente")

Notebook funcionando correctamente


## Spark Session

This section initializes the Spark engine used to process the banking datasets.


In [28]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BankingLakehouse")
    .master("local[*]")
    .getOrCreate()
)

print("Spark session created successfully")
print("Spark version:", spark.version)


Spark session created successfully
Spark version: 4.0.1


## Read source data

The original customer dataset is loaded from the local Raw data directory using PySpark.
No business transformations are applied at this stage.

In [29]:
from pathlib import Path

project_root = Path.cwd().parent
customers_path = project_root / "data" / "raw" / "customers.csv"

print("Project root:", project_root)
print("Customers file:", customers_path)
print("File exists:", customers_path.exists())

Project root: c:\Users\CES\Documents\DATA_ENGINEER\banking-lakehouse-data-engineering
Customers file: c:\Users\CES\Documents\DATA_ENGINEER\banking-lakehouse-data-engineering\data\raw\customers.csv
File exists: True


## Paso 3: leer el CSV con Spark

In [30]:
df_customers_raw = (
    spark.read
    .option("header",True)
    .option("inferSchema",True)
    .option("encoding","UTF-8")
    .csv(str(customers_path))
)

In [31]:
df_customers_raw.show(5, truncate=False)

+---------------+----------+---------+--------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                     |city              |credit_score|created_at|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net  |North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net   |Martinezside      |644         |2020-02-23|
|CUSDD4V30T9NT3W|Randy     |Thompson |trevoranderson@example.org|Gallowayfurt      |670         |2025-06-22|
|CUSGCX1945NQ4FM|Laura     |Phillips |valdezgeorge@example.com  |Morrisview        |573         |2019-10-20|
|CUSVG0FN9XUY41I|Savannah  |Swanson  |smithluis@example.com     |Lake Anna         |332         |2022-07-16|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
only showing top 5 

In [32]:
df_customers_raw.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)



In [33]:
print("Number of rows:", df_customers_raw.count())
print("Number of columns:", len(df_customers_raw.columns))

Number of rows: 50000
Number of columns: 7


In [34]:
df_customers_raw.show(10, truncate=False)


+---------------+----------+---------+--------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                     |city              |credit_score|created_at|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net  |North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net   |Martinezside      |644         |2020-02-23|
|CUSDD4V30T9NT3W|Randy     |Thompson |trevoranderson@example.org|Gallowayfurt      |670         |2025-06-22|
|CUSGCX1945NQ4FM|Laura     |Phillips |valdezgeorge@example.com  |Morrisview        |573         |2019-10-20|
|CUSVG0FN9XUY41I|Savannah  |Swanson  |smithluis@example.com     |Lake Anna         |332         |2022-07-16|
|CUSOC6UZHR5XFF0|Ashley    |Wang     |reesekendra@example.com   |Amybury           |569         |2025-07-22|
|CUSPVN9ER14FFYV|Mo

## Basic data profiling

This section evaluates null values, duplicate business keys, numerical ranges, and date ranges before persisting the Bronze layer.

In [35]:
from pyspark.sql import functions as F

null_counts = df_customers_raw.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in df_customers_raw.columns
])

null_counts.show()

+-----------+----------+---------+-----+----+------------+----------+
|customer_id|first_name|last_name|email|city|credit_score|created_at|
+-----------+----------+---------+-----+----+------------+----------+
|          0|         0|        0|    0|   0|           0|         0|
+-----------+----------+---------+-----+----+------------+----------+



In [36]:
duplicate_customer_ids = (
    df_customers_raw
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", duplicate_customer_ids.count())

Duplicate customer IDs: 0


In [37]:
df_customers_raw.select(
    F.min("credit_score").alias("min_credit_score"),
    F.max("credit_score").alias("max_credit_score"),
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at")
).show()

+----------------+----------------+--------------+--------------+
|min_credit_score|max_credit_score|min_created_at|max_created_at|
+----------------+----------------+--------------+--------------+
|             300|             850|    2019-01-01|    2025-12-31|
+----------------+----------------+--------------+--------------+

